In [26]:
import pandas as pd
import numpy as np
from scipy.stats import norm

In [28]:
parejas = [('Ma', 'Ta'), ('Ma', 'No'), ('Ma', 'Md'), ('Ta', 'No'), ('Ta', 'Md'), ('No', 'Md')]
delta = 0.002
alpha = 0.05

In [41]:
data = pd.read_csv('911_times.csv')

franja_horaria = { # Paso los segundos a años, mentira eso no funcionó
    'Ma': data['mañana'] ,
    'Ta': data['tarde'] ,
    'No': data['noche'] ,
    'Md': data['madrugada'] 
}

def tost(group1, group2, delta, alpha=0.05):
    n1 = len(group1)
    n2 = len(group2)
    
    lambda1 = n1 / np.sum(group1) # tasas λ1 y λ2
    lambda2 = n2 / np.sum(group2)
    # print(f"n1: {n1}, n2: {n2}, λ1: {lambda1}, λ2: {lambda2}")
    
    se = np.sqrt(lambda1**2 / n1 + lambda2**2 / n2) # error estándar combinado
    
    # estadísticos Z_A y Z_B
    Z_A = (lambda1 - lambda2 + delta) / se
    Z_B = (lambda1 - lambda2 - delta) / se
    
    z_alpha = norm.ppf(alpha)
    z_1_minus_alpha = norm.ppf(1 - alpha)
    
    reject_H0 = Z_A > z_1_minus_alpha and Z_B < z_alpha
    
    return Z_A, Z_B, reject_H0

In [42]:
resultados_tost = {}
for bloque1, bloque2 in parejas:
    Z_A, Z_B, reject_H0 = tost(franja_horaria[bloque1], franja_horaria[bloque2], delta, alpha)
    resultados_tost[(bloque1, bloque2)] = (Z_A, Z_B, reject_H0)

for (bloque1, bloque2), (Z_A, Z_B, reject_H0) in resultados_tost.items():
    print(f"Resultado del TOST para ({bloque1}, {bloque2}): Z_A = {Z_A}, Z_B = {Z_B}, Rechazar H0: {reject_H0}")

Resultado del TOST para (Ma, Ta): Z_A = 93506634.06405723, Z_B = -93506154.19863342, Rechazar H0: True
Resultado del TOST para (Ma, No): Z_A = 95546607.145419, Z_B = -95546056.59497315, Rechazar H0: True
Resultado del TOST para (Ma, Md): Z_A = 81815849.41551487, Z_B = -81815718.114831, Rechazar H0: True
Resultado del TOST para (Ta, No): Z_A = 135953117.4320445, Z_B = -135953031.7501936, Rechazar H0: True
Resultado del TOST para (Ta, Md): Z_A = 103166781.89504826, Z_B = -103167145.77177966, Rechazar H0: True
Resultado del TOST para (No, Md): Z_A = 105926300.95037453, Z_B = -105926741.3184319, Rechazar H0: True


In [43]:
resultados_tost_self = {}
for bloque in franja_horaria.keys():
    grupo = franja_horaria[bloque]
    mitad = len(grupo) // 2
    grupo1 = grupo[:mitad]
    grupo2 = grupo[mitad:]
    
    Z_A, Z_B, reject_H0 = tost(grupo1, grupo2, delta, alpha)
    resultados_tost_self[bloque] = (Z_A, Z_B, reject_H0)

for bloque, (Z_A, Z_B, reject_H0) in resultados_tost_self.items():
    print(f"Resultado del TOST para {bloque} consigo misma: Z_A = {Z_A}, Z_B = {Z_B}, Rechazar H0: {reject_H0}")

Resultado del TOST para Ma consigo misma: Z_A = 23175760.933413077, Z_B = -23176533.076369617, Rechazar H0: True
Resultado del TOST para Ta consigo misma: Z_A = 77364500.46539804, Z_B = -77364067.46741623, Rechazar H0: True
Resultado del TOST para No consigo misma: Z_A = 73997289.74672486, Z_B = -73996740.42208344, Rechazar H0: True
Resultado del TOST para Md consigo misma: Z_A = 53557840.55003416, Z_B = -53558246.38000773, Rechazar H0: True
